In [1]:
%pip install pyspark

^C
Note: you may need to restart the kernel to use updated packages.
     ---------------------------------------- 0.0/450.1 MB ? eta -:--:--
     ---------------------------------------- 0.3/450.1 MB ? eta -:--:--
     ---------------------------------------- 0.5/450.1 MB 1.4 MB/s eta 0:05:21
     ---------------------------------------- 0.8/450.1 MB 1.6 MB/s eta 0:04:42
     ---------------------------------------- 1.3/450.1 MB 1.8 MB/s eta 0:04:14
     ---------------------------------------- 1.8/450.1 MB 1.9 MB/s eta 0:03:52
     ---------------------------------------- 2.4/450.1 MB 2.1 MB/s eta 0:03:31
     ---------------------------------------- 3.4/450.1 MB 2.5 MB/s eta 0:03:00
     ---------------------------------------- 4.2/450.1 MB 2.7 MB/s eta 0:02:43
     ---------------------------------------- 5.2/450.1 MB 3.0 MB/s eta 0:02:30
      --------------------------------------- 6.0/450.1 MB 3.1 MB/s eta 0:02:25
      --------------------------------------- 7.3/450.1 MB 3.3 MB

In [13]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [14]:
spark = SparkSession.builder \
    .appName("Week5_Assignment") \
    .master("local[*]") \
    .getOrCreate()

In [15]:
df = spark.read.csv(
    "shopping_data.csv",
    header=True,
    inferSchema=True
)

In [16]:
# shows data till top 10th row
df.show(10)

+-------+----------------+--------------+--------------------+---+------------+------+-----------+----------------+-------+---------+--------+-------------------+
|user_id|transaction_date|      username|               email|age|subscription|region|       city|product_category|  price|   status|store_id|      raw_timestamp|
+-------+----------------+--------------+--------------------+---+------------+------+-----------+----------------+-------+---------+--------+-------------------+
|   1001|      2026-05-30|   Neha Mishra|nehamishra@hotmai...| 18|        Free|  West|       Pune|           Books| 384.19|     NULL|    S101|2026-05-30 07:05:00|
|   1002|      2026-06-10|  Nikhil Verma|nikhilverma@gmail...| 53|     Premium|  East|      Patna|     Electronics|   NULL|  Pending|    S101|2026-06-10 17:54:00|
|   1003|      2026-06-24|     Rohan Das|rohandas@hotmail.com| 56|        Free| North| Chandigarh|         Grocery| 985.56|Completed|    S105|2026-06-24 22:04:00|
|   1004|      2026-05

In [17]:
# print schema is used so pyspark can detect the datatype of each column automatically
df.printSchema()

root
 |-- user_id: integer (nullable = true)
 |-- transaction_date: date (nullable = true)
 |-- username: string (nullable = true)
 |-- email: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- subscription: string (nullable = true)
 |-- region: string (nullable = true)
 |-- city: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- price: double (nullable = true)
 |-- status: string (nullable = true)
 |-- store_id: string (nullable = true)
 |-- raw_timestamp: timestamp (nullable = true)



In [18]:
# prints total number of records and all the columns
print("Total Records:", df.count())
print("Columns:", df.columns)

Total Records: 200
Columns: ['user_id', 'transaction_date', 'username', 'email', 'age', 'subscription', 'region', 'city', 'product_category', 'price', 'status', 'store_id', 'raw_timestamp']


# Q1. What are the key limitations of traditional MapReduce that make Spark a preferred choice for modern big data processing?

In [19]:
#Answer
#.MapReduce stores intermediate results on disk, making it slow.
#.It has high disk I/O overhead.
#.It is inefficient for iterative processing like machine learning.
#.Writing MapReduce programs requires more code.
#.Spark performs computations in memory, making it much faster and easier for modern big data workloads.

# Q2. Explain how Spark uses In-Memory Computing.

In [20]:
#Answer
#Spark uses in-memory computing, which means it stores intermediate data in ram instead of writing it to disk after every operation.
#Since accessing ram is much faster than accessing disk, Spark can process data much more quickly. 
#This is especially useful for iterative tasks like machine learning where the same data is used multiple times.


# Q3.Write a code snippet to remove all duplicate rows from a DataFrame based on a specific set of columns: user_id and transaction_date.

In [21]:
# all the duplicate rows are removed from useid and transaction date
df = df.dropDuplicates(["user_id", "transaction_date"])
df.show()

+-------+----------------+---------------+--------------------+---+------------+------+-----------+----------------+-------+---------+--------+-------------------+
|user_id|transaction_date|       username|               email|age|subscription|region|       city|product_category|  price|   status|store_id|      raw_timestamp|
+-------+----------------+---------------+--------------------+---+------------+------+-----------+----------------+-------+---------+--------+-------------------+
|   1001|      2026-05-30|    Neha Mishra|nehamishra@hotmai...| 18|        Free|  West|       Pune|           Books| 384.19|     NULL|    S101|2026-05-30 07:05:00|
|   1002|      2026-06-10|   Nikhil Verma|nikhilverma@gmail...| 53|     Premium|  East|      Patna|     Electronics|   NULL|  Pending|    S101|2026-06-10 17:54:00|
|   1003|      2026-06-24|      Rohan Das|rohandas@hotmail.com| 56|        Free| North| Chandigarh|         Grocery| 985.56|Completed|    S105|2026-06-24 22:04:00|
|   1004|      2

# Q4. Given a DataFrame df_sales, write a query to filter for rows where the region is 'West' and then group by product_category to find the average sale_amount.

In [24]:
#rows with region west is filtered and grouped by the category of product to find average sales amount
df.filter(col("region") == "West") \
    .groupBy("product_category") \
    .agg(avg("price").alias("average_sale_amount")) \
    .show()

+----------------+-------------------+
|product_category|average_sale_amount|
+----------------+-------------------+
|          Sports|           2685.435|
|         Grocery| 2758.1780000000003|
|     Electronics| 1910.9850000000001|
|        Clothing|            2618.26|
|           Books| 2720.2661538461543|
+----------------+-------------------+



# Q5. What is the difference between .na.drop() and .na.fill()? Provide a code example of filling null values in a status column with the string 'Unknown'.

In [25]:
#.na.drop() removes rows containing null values.
#.na.fill() replaces null values with a specified value.
df = df.na.fill({
    "status":"Unknown"
})

df.show()

+-------+----------------+---------------+--------------------+---+------------+------+-----------+----------------+-------+---------+--------+-------------------+
|user_id|transaction_date|       username|               email|age|subscription|region|       city|product_category|  price|   status|store_id|      raw_timestamp|
+-------+----------------+---------------+--------------------+---+------------+------+-----------+----------------+-------+---------+--------+-------------------+
|   1001|      2026-05-30|    Neha Mishra|nehamishra@hotmai...| 18|        Free|  West|       Pune|           Books| 384.19|  Unknown|    S101|2026-05-30 07:05:00|
|   1002|      2026-06-10|   Nikhil Verma|nikhilverma@gmail...| 53|     Premium|  East|      Patna|     Electronics|   NULL|  Pending|    S101|2026-06-10 17:54:00|
|   1003|      2026-06-24|      Rohan Das|rohandas@hotmail.com| 56|        Free| North| Chandigarh|         Grocery| 985.56|Completed|    S105|2026-06-24 22:04:00|
|   1004|      2

# Q6.Write a query to find the total count of records for each city in a DataFrame, but only for cities where the count is greater than 100.

In [26]:
# there is no city with count greater than 100 since the total dataset is of 200 rows
df.groupBy("city") \
    .count() \
    .filter(col("count") > 100) \
    .show()

+----+-----+
|city|count|
+----+-----+
+----+-----+



# Q7.How does the immutability of Spark DataFrames affect how you perform "data cleaning" steps like dropping columns or renaming them?

In [ ]:
#Answer
#Spark DataFrames are immutable. Operations like dropping columns or renaming columns do not modify the existing DataFrame.
#Instead they create a new DataFrame.

# Q8.Write a Spark command to filter a dataset for rows where the age is between 18 and 30 (inclusive) and the subscription is 'Premium'.

In [27]:
#dataset is filtered for rows of age btwn 18 to 30 including 18 and 30 and subscription is premium
df.filter(
    (col("age").between(18,30)) &
    (col("subscription")=="Premium")
).show()

+-------+----------------+--------------+--------------------+---+------------+------+-----------+----------------+-------+---------+--------+-------------------+
|user_id|transaction_date|      username|               email|age|subscription|region|       city|product_category|  price|   status|store_id|      raw_timestamp|
+-------+----------------+--------------+--------------------+---+------------+------+-----------+----------------+-------+---------+--------+-------------------+
|   1005|      2026-05-12|Ashutosh Yadav|ashutoshyadav@yah...| 25|     Premium|  West|     Mumbai|           Books| 2148.8|Completed|    S101|2026-05-12 17:36:00|
|   1028|      2026-01-04|          NULL|kavyamishra@yahoo...| 28|     Premium| North|      Delhi|        Clothing| 190.33|  Unknown|    S101|2026-01-04 21:52:00|
|   1034|      2026-01-08|  Sourav Sahoo|souravsahoo@gmail...| 22|     Premium|  East|    Kolkata|           Books| 305.12|Completed|    S101|2026-01-08 20:34:00|
|   1039|      2026-06

# Q9. When cleaning a dataset, why is it often better to handle null values before performing mathematical aggregations like sum() or avg()?

In [ ]:
#Answer
#Handling null values before using aggregation functions such as sum() or avg() helps produce accurate and reliable results.
#Null values can lead to missing or misleading calculations if they are not handled appropriately.

# Q10.Write the code to revise a column named raw_timestamp by casting it to a TimestampType and renaming it to event_time.

In [28]:
# the name of the column raw_timestamp is changed or casted to event_time
df = df.withColumn(
    "event_time",
    col("raw_timestamp").cast(TimestampType())
)

df = df.drop("raw_timestamp")

df.show()

+-------+----------------+---------------+--------------------+---+------------+------+-----------+----------------+-------+---------+--------+-------------------+
|user_id|transaction_date|       username|               email|age|subscription|region|       city|product_category|  price|   status|store_id|         event_time|
+-------+----------------+---------------+--------------------+---+------------+------+-----------+----------------+-------+---------+--------+-------------------+
|   1001|      2026-05-30|    Neha Mishra|nehamishra@hotmai...| 18|        Free|  West|       Pune|           Books| 384.19|  Unknown|    S101|2026-05-30 07:05:00|
|   1002|      2026-06-10|   Nikhil Verma|nikhilverma@gmail...| 53|     Premium|  East|      Patna|     Electronics|   NULL|  Pending|    S101|2026-06-10 17:54:00|
|   1003|      2026-06-24|      Rohan Das|rohandas@hotmail.com| 56|        Free| North| Chandigarh|         Grocery| 985.56|Completed|    S105|2026-06-24 22:04:00|
|   1004|      2

# Q11. Explain the "Shuffle" process that occurs during a grouping operation. Why is it considered a wide transformation?

In [ ]:
#Answer
#Shuffle is the process of redistributing data across partitions so that records with the same key are grouped together.
#Operations such as groupBy() require a shuffle. Since data moves between partitions it is called a wide transformation
#making it more expensive than narrow transformations.

# Q12.Write a code snippet that identifies and removes rows where the email column contains null values OR the username is an empty string.

In [32]:
# removes the rows having values in email column null oe empty string
df = df.filter(
    col("email").isNotNull() &
    (trim(col("username")) != "")
)

df.show(10)

+-------+----------------+--------------+--------------------+---+------------+------+----------+----------------+-------+---------+--------+-------------------+
|user_id|transaction_date|      username|               email|age|subscription|region|      city|product_category|  price|   status|store_id|         event_time|
+-------+----------------+--------------+--------------------+---+------------+------+----------+----------------+-------+---------+--------+-------------------+
|   1001|      2026-05-30|   Neha Mishra|nehamishra@hotmai...| 18|        Free|  West|      Pune|           Books| 384.19|  Unknown|    S101|2026-05-30 07:05:00|
|   1002|      2026-06-10|  Nikhil Verma|nikhilverma@gmail...| 53|     Premium|  East|     Patna|     Electronics|   NULL|  Pending|    S101|2026-06-10 17:54:00|
|   1003|      2026-06-24|     Rohan Das|rohandas@hotmail.com| 56|        Free| North|Chandigarh|         Grocery| 985.56|Completed|    S105|2026-06-24 22:04:00|
|   1004|      2026-05-30|  

# Q13.How do you use the .agg() function to calculate multiple statistics at once, such as the min, max, and mean of the price column?

In [30]:
df.agg(
    min("price").alias("Minimum Price"),
    max("price").alias("Maximum Price"),
    avg("price").alias("Average Price")
).show()

+-------------+-------------+-----------------+
|Minimum Price|Maximum Price|    Average Price|
+-------------+-------------+-----------------+
|        145.7|      4991.81|2524.807924528301|
+-------------+-------------+-----------------+



# Q14. In the context of cleaning a dataset, what is the risk of using inferSchema=true when your source data contains messy or inconsistent date formats?

In [ ]:
#Answer
#When the source data contains inconsistent or messy date formats, (inferSchema=True) may infer the wrong data type or treat date columns as strings.
#This can lead to parsing errors or incorrect results during analysis.

# Q15.Write a final processing pipeline that: 

# Filters out duplicates. 

# Fills null prices with 0. 

# Groups by store_id to calculate total revenue. 

In [31]:
# here all duplicate values are filtered and null values are filled with zeroes and grouped by store_id to calculate the total revenue
pipeline = (
    df
    .dropDuplicates()
    .na.fill({"price":0})
    .groupBy("store_id")
    .agg(
        sum("price").alias("total_revenue")
    )
)

pipeline.show()

+--------+------------------+
|store_id|     total_revenue|
+--------+------------------+
|    S105| 80494.16000000002|
|    S102| 88857.20999999998|
|    S104| 94841.02000000002|
|    S101|61645.009999999995|
|    S103|          75607.06|
+--------+------------------+

